# 02 - 9-Level Strategy Chain in Detail

> **When to use**: When you need to customize the data type per column, or are unsatisfied with auto-inferred results.
>
> **Core concept**: sqlseed's `ColumnMapper` auto-matches column names → generators by 9-level priority. Understanding this chain lets you precisely control each column's data.

## Applicable Scenarios

- Column names are non-standard (e.g., `user_name` instead of `name`), need manual generator assignment
- Need to limit data range (e.g., `age` between 18-65)
- Need to generate specific pattern data (e.g., order number `ORD-\d{6}`)
- Want to understand sqlseed's auto-inference logic

## What You Will Learn

- Complete priority of the 9-level strategy chain
- Matching logic and trigger conditions for each level
- 74 exact match rules + 26 regex patterns
- How to override auto-inference with `columns={}`

See architecture.zh-CN.md §3

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| **→ 02** | **9-Level Strategy Chain** | **Core: ColumnMapper** | **01** |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Schema Inference | `src/sqlseed/core/schema.py` | `SchemaInferrer` |
| Column Mapping | `src/sqlseed/core/mapper.py` | `ColumnMapper.map_column()` |

> Corresponding architecture diagram: [§3 ColumnMapper 9-Level Strategy Chain](../docs/architecture.zh-CN.md#3-columnmapper-9-级策略链)

## 1. See the Effect First — The Magic of Zero Config

sqlseed's most powerful feature: **without any configuration**, it can infer the correct data type from column names. Column named `email`? Generate an email. Column named `name`? Generate a name. Column named `created_at`? Generate a timestamp.

See the effect first, then explain the principle:

In [2]:
# Zero config! sqlseed auto-selects generators based on column names
rows = preview(str(db_path), table="members", count=3)
print(f"{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'org_code':<10s}")
print('-' * 78)
for row in rows:
    print(f"{row.get('name', 'N/A'):<18s}  {row.get('email', 'N/A'):<28s}  {row.get('phone', 'N/A'):<18s}  {row.get('org_code', 'N/A'):<10s}")  # noqa: E501

name                email                         phone               org_code  
------------------------------------------------------------------------------
Scott Mcmahon       committed2091@example.org     +1-386-559-5630     fLBcbfnoGM
Claud Reese         settings1825@example.com      +17728572576        JmTPSI    
Neil Mercer         ridge2025@duck.com            +18653406653        fLBcbfnoGM


No mapping config written — sqlseed's `ColumnMapper` auto-completed:

- `name` column → matches `name` rule → generates real name
- `email` column → matches `email` rule → generates email address
- `phone` column → matches `phone` rule → generates phone number
- `member_no` column → matches UNIQUE constraint → generates unique ID

The secret behind this is the **9-level strategy chain** — sqlseed tries each level by priority until it finds a matching generator.

## 2. Strategy Chain Overview

sqlseed's `ColumnMapper` tries to match column names in the following priority order:

| Level | Strategy | Description |
|:----:|------|------|
| 1 | Autoincrement PK | Auto-increment PK auto-skipped |
| 2 | User Config | User explicit config overrides all |
| 3 | Custom Exact Match | Plugin-registered exact rules |
| 4 | Built-in Exact Match | 74 built-in exact match rules |
| 5 | DEFAULT Value | Columns with default skipped or enriched |
| 6 | Custom Pattern Match | Plugin-registered regex rules |
| 7 | Built-in Pattern Match | 26 built-in regex pattern matches |
| 8 | Nullable | Nullable columns skipped or enriched |
| 9 | Type Fallback | 22 SQL types faithful fallback |

Once a level matches, subsequent levels are not executed. Levels 3 and 6 are plugin extension points, see 08-plugin-hooks.ipynb.

## 3. Level 1: Autoincrement PK

If the column is a primary key and auto-increment (or type INTEGER/INT), it's auto-skipped, no data generated.

In [3]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=2)
for row in rows:
    print(f"name={row['name']}, email={row['email']}")
print("\nmember_id is an auto-increment PK, preview does not include this column (auto-assigned by SQLite)")

name=Lien Duke, email=carb1837@protonmail.com
name=Willodean Hoover, email=cube2042@duck.com

member_id is an auto-increment PK, preview does not include this column (auto-assigned by SQLite)


## 4. Level 2: User Config

Users explicitly specify generators via the `columns` parameter or YAML config, with the highest priority (only after auto-increment PK skip).

In [4]:
rows = preview(
    str(db_path),
    table="members",
    count=2,
    columns={
        "name": {"generator": "pattern", "params": {"regex": "User-\\d{4}"}},
        "balance": {"generator": "float", "params": {"min_value": 1000.0, "max_value": 5000.0}},
    },
)
for row in rows:
    print(f"name={row['name']}, balance={row['balance']}")
print("\nname and balance are overridden by user config, no longer using auto-inference")

name=User-3435, balance=4686.8
name=User-7312, balance=3861.57

name and balance are overridden by user config, no longer using auto-inference


## 5. Level 4: Built-in Exact Match (74 rules)

Exact match is the most commonly used strategy. sqlseed has 74 built-in column-name-to-generator mapping rules.

### Semantic (infer business meaning)

| Column Name | Generator | Params |
|------|--------|------|
| `email` | email | - |
| `phone` | phone | - |
| `name` | name | - |
| `address` | address | - |
| `city` | city | - |
| `country` | country | - |
| `url` / `website` | url | - |
| `password` | password | - |
| `uuid` | uuid | - |

### Numeric (with reasonable range)

| Column Name | Generator | Params |
|------|--------|------|
| `age` | integer | 18-65 |
| `balance` | float | 0-999999.99 |
| `salary` | float | 3000-100000 |
| `rating` | float | 1.0-5.0 |
| `latitude` | float | -90~90 |

### Enum (fixed options)

| Column Name | Generator | Params |
|------|--------|------|
| `status` | choice | [0, 1] |
| `gender` | choice | ["male", "female", "other"] |
| `priority` | choice | ["low", "medium", "high"] |
| `role` | choice | ["admin", "user", "guest"] |

In [5]:
rows = preview(str(db_path), table="members", count=3)
for row in rows:
    print(f"name={row['name']}, email={row['email']}, phone={row['phone']}, balance={row['balance']}")
print("\nname/email/phone/balance all auto-inferred via exact match")

name=Monroe Moran, email=terminal2034@yandex.com, phone=+17308896671, balance=221146.24
name=Erik Sampson, email=medium1840@yahoo.com, phone=+17083994550, balance=623384.49
name=Len Burton, email=pmid1935@gmail.com, phone=+17735220223, balance=100396.55

name/email/phone/balance all auto-inferred via exact match


## 6. Level 5: DEFAULT Value

If a column has a DEFAULT value, sqlseed skips it by default (uses the default value). When `enrich=True`, it analyzes the default value pattern.

In [6]:
import sqlite3

conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(projects)").fetchall()
default_cols = [c[1] for c in cols if c[4] is not None]
print(f"Columns with DEFAULT: {default_cols}")
conn.close()

rows = preview(str(db_path), table="projects", count=2)
print(f"preview output columns: {list(rows[0].keys())}")
skipped = [c for c in default_cols if c not in rows[0]]
print(f"Skipped DEFAULT columns: {skipped}")
print("\nColumns with DEFAULT are skipped by default, using default values (no data generated)")

Columns with DEFAULT: ['budget', 'task_count', 'is_public', 'is_archived']
preview output columns: ['project_no', 'short_code', 'name', 'org_code', 'created_at', 'description']
Skipped DEFAULT columns: ['budget', 'task_count', 'is_public', 'is_archived']

Columns with DEFAULT are skipped by default, using default values (no data generated)


## 7. Level 7: Built-in Pattern Match (26 regex)

> **Note**: Level 3 (Custom Exact Match) and Level 6 (Custom Pattern Match) are plugin extension points, register custom rules via the `sqlseed_register_column_mappers` Hook. See [08-plugin-hooks.ipynb](08-plugin-hooks.ipynb) Section 10.

When exact match fails, sqlseed uses regex patterns to match column name suffixes:

| Pattern | Generator | Example Column |
|------|--------|----------|
| `.*_id$` | foreign_key_or_integer | `project_id`, `assignee_id` |
| `.*_no$` / `.*_nbr$` | foreign_key_or_integer | `project_no`, `member_no` |
| `.*_at$` | datetime | `created_at`, `due_at` |
| `.*_date$` | date | `birth_date` |
| `^is_.*` / `^has_.*` | boolean | `is_active`, `is_public` |
| `.*_code$` | string (alphanumeric) | `org_code`, `region_code` |
| `.*_name$` | name | `org_name`, `file_name` |
| `.*_count$` / `.*_num$` | integer (0-10000) | `task_count`, `item_num` |
| `.*_amount$` / `.*_price$` | float | `total_amount`, `unit_price` |

In [7]:
# *_no pattern match → foreign_key_or_integer → no FK constraint, falls back to random string (because it's VARCHAR)
# org_code matches *_code → should be string(alphanumeric), but auto-detected FK constraint, smartly upgraded to foreign_key and extracted existing real values from organizations!  # noqa: E501
# *_at pattern match → datetime
rows = preview(str(db_path), table="projects", count=3)
for row in rows:
    pno = str(row['project_no'])[:20]
    code = str(row['org_code'])[:12]
    print(f"project_no={pno:<20s}  org_code={code:<12s}  created_at={row['created_at']}")
print("\nproject_no → *_no pattern → foreign_key_or_integer (no FK, falls back to random string)")
print("org_code → auto-detected FK constraint, smartly upgraded to foreign_key (extracted real values from parent table)")
print("created_at → *_at pattern → datetime")

project_no=NzOES4pWSuBNM         org_code=JmTPSI        created_at=2024-09-20 21:42:26.229719
project_no=OMlZdFi               org_code=oCLrZ3aWZ     created_at=2002-04-19 03:50:52.291483
project_no=2Dn4aMrws1hzXb        org_code=hbVrpoiVgRV   created_at=2006-05-30 07:58:09.416734

project_no → *_no pattern → foreign_key_or_integer (no FK, falls back to random string)
org_code → auto-detected FK constraint, smartly upgraded to foreign_key (extracted real values from parent table)
created_at → *_at pattern → datetime


## 8. Level 8: Nullable

Nullable columns (without DEFAULT values) are skipped by default. When `enrich=True`, generation strategy is inferred from existing data.

In [8]:
conn = sqlite3.connect(str(db_path))
cols = conn.execute("PRAGMA table_info(members)").fetchall()
nullable_cols = [c[1] for c in cols if c[3] == 0 and c[4] is None]
print(f"Nullable columns (no default): {nullable_cols}")
conn.close()

Nullable columns (no default): ['member_id', 'phone', 'avatar', 'registered_at', 'address']


## 9. Level 9: Type Fallback (22 SQL types)

When all naming strategies fail, sqlseed selects a generator based on the column's SQL type:

| SQL Type | Generator | Params |
|----------|--------|------|
| INTEGER | integer | 0-999999 |
| REAL / FLOAT | float | 0-999999 |
| TEXT | string | 5-50 chars |
| VARCHAR(n) | string | 1-n chars |
| BLOB | bytes | 32 bytes |
| BOOLEAN | boolean | - |
| DATE | date | - |
| DATETIME | datetime | - |

In [9]:
conn = sqlite3.connect(str(db_path))
conn.execute("""CREATE TABLE IF NOT EXISTS demo_fallback (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    info TEXT NOT NULL,
    score_value REAL NOT NULL
)""")
conn.commit()
conn.close()

rows = preview(str(db_path), table="demo_fallback", count=2)
for row in rows:
    info = str(row['info'])[:50]
    print(f"info=\"{info}...\", score_value={row['score_value']:.2f}")
print("\ninfo (TEXT, no column name match) → type fallback → random string (5-50 chars)")
print("score_value (REAL, no column name match) → type fallback → random float")

# Cleanup
conn = sqlite3.connect(str(db_path))
conn.execute("DROP TABLE IF EXISTS demo_fallback")
conn.commit()
conn.close()

info="7VzI5SB1_qPez2SuTjxv8uu4hf6...", score_value=956096.77
info="OjkddN4RzhMOgLNPzNDXdKvCBfz5uojo...", score_value=548039.88

info (TEXT, no column name match) → type fallback → random string (5-50 chars)
score_value (REAL, no column name match) → type fallback → random float


## 10. inspect --show-mapping in Practice

Use the CLI's `inspect --show-mapping` command to view the mapping result for each column:

In [10]:
from click.testing import CliRunner

from sqlseed.cli.main import cli

runner = CliRunner()
result = runner.invoke(cli, ["inspect", str(db_path), "--show-mapping"])
if result.output.strip():
    print(result.output)

                                           Table: organizations (5 rows)                                           
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column       ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                        ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ org_code     │ VARCHAR(16) │ ✗        │ ✓  │      │ string      │ {'min_length': 6, 'max_length': 12,           │
│              │             │          │    │      │             │ 'charset': 'alphanumeric'}                    │
│ name         │ VARCHAR(64) │ ✗        │    │      │ name        │ {}                                            │
│ parent_code  │ VARCHAR(16) │ ✓        │    │      │ foreign_key │ {'ref_table': 'organizations', 'ref_column':  │
│              │             │          │    │      │             │ 'org_code', 'strategy': 'random',             │
│              │             │          │    │      │             │ '_ref_values': ['JmTPSI', 'SBvrjn9',          │
│              │             │          │    │      │             │ 'fLBcbfnoGM', 'hbVrpoiVgRV', 'oCLrZ3aWZ']}    │
│ description  │ TEXT        │ ✓        │    │      │ text        │ {'min_length': 100, 'max_length': 500}        │
│ is_active    │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ member_count │ INTEGER     │ ✓        │    │      │ skip        │ {}                                            │
│ created_at   │ TEXT        │ ✓        │    │      │ datetime    │ {}                                            │
└──────────────┴─────────────┴──────────┴────┴──────┴─────────────┴───────────────────────────────────────────────┘

        Foreign Keys: organizations         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column      ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ parent_code │ organizations │ org_code   │
└─────────────┴───────────────┴────────────┘

                                             Table: members (20 rows)                                              
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column        ┃ Type         ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                      ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ member_id     │ INTEGER      │ ✗        │ ✓  │ ✓    │ skip        │ {}                                          │
│ member_no     │ VARCHAR(16)  │ ✗        │    │      │ string      │ {'min_length': 4, 'max_length': 20}         │
│ name          │ VARCHAR(64)  │ ✗        │    │      │ name        │ {}                                          │
│ email         │ VARCHAR(128) │ ✗        │    │      │ email       │ {}                                          │
│ phone         │ VARCHAR(20)  │ ✓        │    │      │ phone       │ {}                                          │
│ org_code      │ VARCHAR(16)  │ ✗        │    │      │ foreign_key │ {'ref_table': 'organizations',              │
│               │              │          │    │      │             │ 'ref_column': 'org_code', 'strategy':       │
│               │              │          │    │      │             │ 'random', '_ref_values': ['JmTPSI',         │
│               │              │          │    │      │             │ 'SBvrjn9', 'fLBcbfnoGM', 'hbVrpoiVgRV',     │
│               │              │          │    │      │             │ 'oCLrZ3aWZ']}                               │
│ is_active     │ INTEGER      │ ✓        │    │      │ skip        │ {}                                          │
│ balance       │ REAL         │ ✓        │    │      │ float       │ {'min_value': 0.0, 'max_value': 999999.99,  │
│               │              │          │    │      │             │ 'precision': 2}                             │
│ avatar        │ BLOB         │ ✓        │    │      │ url         │ {}                                          │
│ registered_at │ TEXT         │ ✓        │    │      │ datetime    │ {}                                          │
│ address       │ TEXT         │ ✓        │    │      │ address     │ {}                                          │
└───────────────┴──────────────┴──────────┴────┴──────┴─────────────┴─────────────────────────────────────────────┘

          Foreign Keys: members          
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column   ┃ Ref Table     ┃ Ref Column ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ org_code │ organizations │ org_code   │
└──────────┴───────────────┴────────────┘

               Table: sqlite_sequence (3 rows)               
┏━━━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Column ┃ Type ┃ Nullable ┃ PK ┃ Auto ┃ Generator ┃ Params ┃
┡━━━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ name   │      │ ✓        │    │      │ name      │ {}     │
│ seq    │      │ ✓        │    │      │ skip      │ {}     │
└────────┴──────┴──────────┴────┴──────┴───────────┴────────┘

...

                          Table: tags (8 rows)                           
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Column      ┃ Type        ┃ Nullable ┃ PK ┃ Auto ┃ Generator ┃ Params ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ tag_id      │ INTEGER     │ ✗        │ ✓  │ ✓    │ skip      │ {}     │
│ name        │ VARCHAR(32) │ ✗        │    │      │ name      │ {}     │
│ color       │ VARCHAR(7)  │ ✓        │    │      │ skip      │ {}     │
│ usage_count │ INTEGER     │ ✓        │    │      │ skip      │ {}     │
└─────────────┴─────────────┴──────────┴────┴──────┴───────────┴────────┘

                                             Table: task_tags (0 rows)                                             
┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column  ┃ Type    ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                                 ┃
┡━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ task_id │ INTEGER │ ✗        │    │      │ foreign_key │ {'ref_table': 'tasks', 'ref_column': 'task_id',        │
│         │         │          │    │      │             │ 'strategy': 'random', '_ref_values': []}               │
│ tag_id  │ INTEGER │ ✗        │    │      │ foreign_key │ {'ref_table': 'tags', 'ref_column': 'tag_id',          │
│         │         │          │    │      │             │ 'strategy': 'random', '_ref_values': [1, 5, 3, 6, 8,   │
│         │         │          │    │      │             │ 2, 7, 4]}                                              │
└─────────┴─────────┴──────────┴────┴──────┴─────────────┴────────────────────────────────────────────────────────┘

      Foreign Keys: task_tags       
┏━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column  ┃ Ref Table ┃ Ref Column ┃
┡━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ tag_id  │ tags      │ tag_id     │
│ task_id │ tasks     │ task_id    │
└─────────┴───────────┴────────────┘

                                            Table: attachments (0 rows)                                            
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column        ┃ Type         ┃ Nullable ┃ PK ┃ Auto ┃ Generator   ┃ Params                                      ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ attachment_id │ INTEGER      │ ✗        │ ✓  │ ✓    │ skip        │ {}                                          │
│ task_id       │ INTEGER      │ ✗        │    │      │ foreign_key │ {'ref_table': 'tasks', 'ref_column':        │
│               │              │          │    │      │             │ 'task_id', 'strategy': 'random',            │
│               │              │          │    │      │             │ '_ref_values': []}                          │
│ file_name     │ VARCHAR(128) │ ✗        │    │      │ name        │ {}                                          │
│ file_data     │ BLOB         │ ✓        │    │      │ skip        │ {}                                          │
│ file_size     │ INTEGER      │ ✓        │    │      │ skip        │ {}                                          │
│ uploaded_at   │ TEXT         │ ✓        │    │      │ datetime    │ {}                                          │
└───────────────┴──────────────┴──────────┴────┴──────┴─────────────┴─────────────────────────────────────────────┘

     Foreign Keys: attachments      
┏━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Column  ┃ Ref Table ┃ Ref Column ┃
┡━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ task_id │ tasks     │ task_id    │
└─────────┴───────────┴────────────┘

## 11. Custom Mapping Rules (Level 3 / Level 6)

Custom rules can be registered via the `sqlseed_register_column_mappers` plugin Hook. These rules take priority over built-in rules in Level 3 (exact match) and Level 6 (pattern match):

In [11]:
import pluggy

from sqlseed.plugins.hookspecs import SqlseedHookSpec, hookimpl

pm = pluggy.PluginManager("sqlseed")
pm.add_hookspecs(SqlseedHookSpec)

class CustomMapperPlugin:
    @hookimpl
    def sqlseed_register_column_mappers(self, mapper):
        mapper.register_exact_rule("color", "choice", {"choices": ["red", "green", "blue"]})
        mapper.register_pattern_rule(r".*_color$", "choice", {"choices": ["#ff0000", "#00ff00", "#0000ff"]})

pm.register(CustomMapperPlugin())
print("Custom mapper plugin registered")
print("  Exact rule: 'color' → choice([red, green, blue])")
print("  Pattern rule: '*_color' → choice([#ff0000, #00ff00, #0000ff])")

Custom mapper plugin registered
  Exact rule: 'color' → choice([red, green, blue])
  Pattern rule: '*_color' → choice([#ff0000, #00ff00, #0000ff])


## 🎯 enrich Mode: __enrich__ Behavior for Level 5 and Level 8

Normally, columns in Level 5 (with DEFAULT values) and Level 8 (nullable) are **skipped**. But when `enrich=True`, these columns enter `__enrich__` mode:

- **DEFAULT columns**: if identified as enum columns by EnrichmentEngine, generate meaningful enum values
- **Nullable columns**: if identified as enum columns, generate enum values; otherwise generate based on null_ratio

### EnrichmentEngine Enum Column Detection

EnrichmentEngine uses 19 column name patterns to detect enum columns:

| Pattern | Example Column |
|---|---|
| `*_status` | order_status, project_status |
| `*_type` | user_type, file_type |
| `is_*` | is_active, is_public |
| `has_*` | has_permission |
| `*_level` | priority_level, access_level |
| `*_category` | product_category |
| `*_flag` | feature_flag |
| `*_mode` | payment_mode |
| ... | 19 patterns in total |

Additionally, it uses **cardinality ratio** (distinct_count / total_rows < 0.3) and **small integer types** (INT8/INT16/TINYINT/SMALLINT) to assist in judgment.

In [12]:
from sqlseed.core.enrichment import EnrichmentEngine

print("EnrichmentEngine enum column name patterns (19):")
for i, pattern in enumerate(EnrichmentEngine.ENUM_NAME_PATTERNS, 1):
    print(f"  {i:2d}. {pattern}")

print(f"\nSmall integer types: {EnrichmentEngine.SMALL_INT_TYPES}")

EnrichmentEngine enum column name patterns (19):
   1. ^[bB]y[A-Za-z]
   2. .*_type$
   3. .*_status$
   4. ^is_.*
   5. ^has_.*
   6. ^can_.*
   7. .*_level$
   8. .*_category$
   9. .*_class$
  10. .*_flag$
  11. .*_kind$
  12. .*_grade$
  13. .*_rank$
  14. .*_tier$
  15. .*_mode$
  16. .*_stage$
  17. .*_phase$
  18. .*_state$
  19. .*_group$

Small integer types: ('INT8', 'INT16', 'TINYINT', 'SMALLINT')


## 🔗 foreign_key_or_integer Smart Resolution

In Level 7, `*_id` and `*_no` columns match the `foreign_key_or_integer` generator. Its resolution logic:

1. **Check FK constraint first**: if the column has a `FOREIGN KEY` constraint → use `foreign_key` generator
2. **Check SharedPool**: if SharedPool has values for the same column name → use `foreign_key` generator
3. **Fallback by column type**: INTEGER type → `integer` generator, others → `string` generator

In [13]:
with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("members")
    fk_info = orch.get_foreign_keys("members")

    fk_columns = {fk.column for fk in fk_info}
    print("members table column mapping analysis:")
    for col in col_info:
        if col.name.endswith(("_id", "_no")):
            is_fk = col.name in fk_columns
            gen_type = "foreign_key" if is_fk else "integer/string"
            print(f"  {col.name}: {gen_type} (FK={is_fk})")

members table column mapping analysis:
  member_id: integer/string (FK=False)
  member_no: integer/string (FK=False)


## 12. Summary

| Level | Strategy | Rule Count | Priority |
|:----:|------|:------:|:------:|
| 1 | Autoincrement PK | - | Highest |
| 2 | User Config | Unlimited | Very High |
| 3 | Custom Exact Match | Plugin-registered | High |
| 4 | Built-in Exact Match | 74 | High |
| 5 | DEFAULT Value | - | Medium-High |
| 6 | Custom Pattern Match | Plugin-registered | Medium |
| 7 | Built-in Pattern Match | 26 | Medium |
| 8 | Nullable | - | Medium-Low |
| 9 | Type Fallback | 22 | Lowest |

**Key Insights**:
- Naming is more important than type — `email VARCHAR(128)` generates an email, not a random string
- User config can override all auto-inference
- Plugins can inject custom rules via Level 3/6, taking priority over built-in rules
- `inspect --show-mapping` is the primary tool for debugging mapping issues

**Next**: [03-generators.ipynb](03-generators.ipynb) — Learn about 31 generators and the Provider system

In [14]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
